# J0_plan (local CPU): seed list, chunks and US-input check for the Japanese-culture evaluation (EN arm)

In [1]:
ARM = "en"


In [2]:
# Parameters
ARM = "en"


In [3]:
import glob, hashlib, json, os, time
from pathlib import Path
import pandas as pd

MYDRIVE = Path(glob.glob(os.path.expanduser("~/Library/CloudStorage/GoogleDrive-*"))[0]) / "My Drive"
PT3 = MYDRIVE / "ez-penta" / "post_train_3"
JC = PT3 / "japan_culture"
JC.mkdir(parents=True, exist_ok=True)


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def load_jsonl(p):
    return [json.loads(l) for l in Path(p).read_text(encoding="utf-8").splitlines() if l.strip()]

assert not (JC / "plan.DONE").exists(), "plan.DONE exists: the plan is written once and never changed"


In [4]:
# 1. Verify batch DONE hashes and the frozen pool.
frozen_path = PT3 / "arm_en" / "frozen_pool.json"
frozen = json.loads(frozen_path.read_text())
FROZEN_SHA = sha256_file(frozen_path)
BATCHES = sorted(frozen["batches"])
assert BATCHES == [1, 3, 5, 7, 9, 11], BATCHES
batch_manifest_sha = {}
for n in BATCHES:
    bd = PT3 / "eval_pool" / "batches" / f"b{n:04d}"
    assert (bd / "DONE").read_text().strip() == sha256_file(bd / "manifest.json"), f"b{n:04d}: DONE != sha256(manifest)"
    man = json.loads((bd / "manifest.json").read_text())
    for f, meta in man["files"].items():
        assert sha256_file(bd / f) == meta["sha256"], f"b{n:04d}/{f} hash mismatch"
    batch_manifest_sha[n] = sha256_file(bd / "manifest.json")
print("batches verified:", BATCHES, "| frozen_pool.json sha256", FROZEN_SHA[:16])


batches verified: [1, 3, 5, 7, 9, 11] | frozen_pool.json sha256 7370deeeb8e0859e


In [5]:
# 2. Seed list (first occurrence in ascending batch / row order) and chunks.
fset = set(frozen["seed_ids"])
seeds, seen = [], set()
for n in BATCHES:
    for r in load_jsonl(PT3 / "eval_pool" / "batches" / f"b{n:04d}" / "seeds.jsonl"):
        if r["seed_id"] in fset and r["seed_id"] not in seen:
            seen.add(r["seed_id"])
            seeds.append({"seed_id": r["seed_id"], "chunk": n, "row": r["row"], "category": r["category"], "severity": r["severity"]})
per = pd.Series([s["chunk"] for s in seeds]).value_counts().sort_index().to_dict()
print(len(seeds), per)
assert len(seeds) == 519 and per == {1: 97, 3: 89, 5: 90, 7: 82, 9: 81, 11: 80}, per

# US prompt IDs per seed, from the seed's own (first) batch.
us_pids = {}
for n in BATCHES:
    want = {s["seed_id"] for s in seeds if s["chunk"] == n}
    for p in load_jsonl(PT3 / "eval_pool" / "batches" / f"b{n:04d}" / "prompts.jsonl"):
        if p["seed_id"] in want:
            us_pids.setdefault(p["seed_id"], {})[f"{p['kind']}|{p['framing']}|{p['lang']}"] = p["prompt_id"]
assert all(len(us_pids[s["seed_id"]]) == 12 for s in seeds)


519 {1: 97, 3: 89, 5: 90, 7: 82, 9: 81, 11: 80}


In [6]:
# 3. US inputs for reuse: arm_en eval rows for base and en_s0, both judges, all 12 US prompts of every seed.
us_sources = {}
for n in BATCHES:
    ev = PT3 / "arm_en" / "eval" / f"b{n:04d}.jsonl"
    em = json.loads((PT3 / "arm_en" / "eval" / f"b{n:04d}.manifest.json").read_text())
    assert sha256_file(ev) == em["sha256"], f"eval b{n:04d} hash mismatch"
    df = pd.read_json(ev, lines=True)
    want = {pid for s in seeds if s["chunk"] == n for pid in us_pids[s["seed_id"]].values()}
    for m in ("base", "en_s0"):
        sub = df[(df.model == m) & df.prompt_id.isin(want)]
        assert set(sub.prompt_id) == want, (n, m, len(want - set(sub.prompt_id)))
        assert sub.g06_label.notna().all() and sub.g4b_label.notna().all()
    us_sources[n] = {"eval_sha256": em["sha256"], "hardware": em["hardware"], "dtype": em["dtype"], "versions": em["versions"],
                     "serving": em["serving"], "base_reused_from_batch": em["base_reused_from_batch"]}
print(json.dumps(us_sources[1], indent=1))


{
 "eval_sha256": "f89870f51970a96b0024f3c9583e7513f45bfc559ec5cc893db0c271aa89ef88",
 "hardware": "G4",
 "dtype": "bfloat16",
 "versions": {
  "torch": "2.14.0+cu130",
  "vllm": "0.33.0",
  "transformers": "5.20.2",
  "peft": "0.21.2",
  "python": "3.13.16"
 },
 "serving": "merged",
 "base_reused_from_batch": true
}


In [7]:
# 4. Write plan.json, then plan.DONE.
seed_list_sha = hashlib.sha256("\n".join(sorted(s["seed_id"] for s in seeds)).encode()).hexdigest()
plan = {"created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "writer": "en",
        "frozen_pool_sha256": FROZEN_SHA, "batch_manifest_sha256": batch_manifest_sha,
        "chunks": BATCHES, "seeds": seeds, "us_prompt_ids": us_pids, "us_answer_sources": us_sources,
        "seed_list_sha256": seed_list_sha, "n_seeds": len(seeds), "seeds_per_chunk": per}
(JC / "plan.json").write_text(json.dumps(plan, indent=1))
(JC / "plan.DONE").write_text(sha256_file(JC / "plan.json") + "\n")
print("plan written; seed list sha256", seed_list_sha[:16])


plan written; seed list sha256 3643b0a7941d8a54
